In [ ]:
import os

PROJECT_DIR = "/content/ResearchMind_AI"

folders = [
    "agents",
    "rag",
    "workflow",
    "uploads",
    "exports",
    ".streamlit"
]

for folder in folders:
    os.makedirs(
        os.path.join(PROJECT_DIR, folder),
        exist_ok=True
    )

print("PROJECT STRUCTURE")
print("=" * 40)

for folder in folders:
    print(folder, "READY")

print("\nSTEP 1 COMPLETED")

PROJECT STRUCTURE
agents READY
rag READY
workflow READY
uploads READY
exports READY
.streamlit READY

STEP 1 COMPLETED


In [ ]:
!pip install -q \
    streamlit \
    google-genai \
    tavily-python \
    langgraph \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-docx \
    reportlab \
    pandas \
    numpy

In [ ]:
import streamlit
import google.genai
import tavily
import langgraph
import faiss
import sentence_transformers
import pypdf
import docx
import reportlab

print("All required libraries imported successfully.")

All required libraries imported successfully.


In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

assert os.environ["GEMINI_API_KEY"]
assert os.environ["TAVILY_API_KEY"]

print("Gemini API: CONFIGURED")
print("Tavily API: CONFIGURED")

Gemini API: CONFIGURED
Tavily API: CONFIGURED


In [ ]:
import os

PROJECT_DIR = "/content/ResearchMind_AI"

print("========== RUNTIME CHECK ==========")
print("Project folder exists:", os.path.isdir(PROJECT_DIR))

if os.path.isdir(PROJECT_DIR):
    print("\nFiles and folders:")
    for item in os.listdir(PROJECT_DIR):
        path = os.path.join(PROJECT_DIR, item)

        if os.path.isfile(path):
            print(f"FILE   : {item}")
        else:
            print(f"FOLDER : {item}")
else:
    print("\nResearchMind_AI folder is missing.")
    print("Colab runtime may have reset.")

========== RUNTIME CHECK ==========
Project folder exists: True

Files and folders:
FOLDER : workflow
FOLDER : rag
FOLDER : exports
FOLDER : .streamlit
FOLDER : uploads
FOLDER : agents


In [ ]:
import ast

required_names = [
    "GeminiService",
    "ResearchSearch",
    "OrchestratorAgent",
    "ResearchAgent",
    "AnalysisAgent",
    "FactCheckerAgent",
    "SummarizerAgent",
    "WriterAgent",
    "QualityAgent",
    "DocumentLoader",
    "TextChunker",
    "FAISSVectorStore",
    "ResearchRAG",
    "ResearchState",
    "orchestrator_node",
    "research_node",
    "analysis_node",
    "fact_checker_node",
    "summarizer_node",
    "writer_node",
    "quality_node",
    "quality_decision",
    "revision_node"
]

source_map = {}

for cell in reversed(In):
    if not isinstance(cell, str):
        continue

    try:
        tree = ast.parse(cell)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, (
            ast.ClassDef,
            ast.FunctionDef,
            ast.AsyncFunctionDef
        )):
            if node.name in required_names and node.name not in source_map:
                source_map[node.name] = ast.unparse(node)

print("========== BACKEND CODE RECOVERY ==========")
print("Components found:", len(source_map), "/", len(required_names))

for name in required_names:
    status = "FOUND" if name in source_map else "MISSING"
    print(f"{name}: {status}")

missing = [name for name in required_names if name not in source_map]

print("\nRecovery status:", "READY" if not missing else "INCOMPLETE")

========== BACKEND CODE RECOVERY ==========
Components found: 2 / 23
GeminiService: MISSING
ResearchSearch: MISSING
OrchestratorAgent: MISSING
ResearchAgent: MISSING
AnalysisAgent: MISSING
FactCheckerAgent: FOUND
SummarizerAgent: MISSING
WriterAgent: FOUND
QualityAgent: MISSING
DocumentLoader: MISSING
TextChunker: MISSING
FAISSVectorStore: MISSING
ResearchRAG: MISSING
ResearchState: MISSING
orchestrator_node: MISSING
research_node: MISSING
analysis_node: MISSING
fact_checker_node: MISSING
summarizer_node: MISSING
writer_node: MISSING
quality_node: MISSING
quality_decision: MISSING
revision_node: MISSING

Recovery status: INCOMPLETE


In [ ]:
%%writefile /content/ResearchMind_AI/researchmind_core.py

import os
import json
import numpy as np
import faiss

from typing import TypedDict, List, Dict, Any
from google import genai
from tavily import TavilyClient
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from docx import Document
from langgraph.graph import StateGraph, START, END


# =====================================================
# 1. CONFIGURATION
# =====================================================

GEMINI_MODEL = "gemini-3.5-flash-lite"

if not os.environ.get("GEMINI_API_KEY"):
    raise EnvironmentError("GEMINI_API_KEY is missing.")

if not os.environ.get("TAVILY_API_KEY"):
    raise EnvironmentError("TAVILY_API_KEY is missing.")


# =====================================================
# 2. GEMINI SERVICE
# =====================================================

class GeminiService:

    def __init__(self, client, model):
        self.client = client
        self.model = model

    def generate(self, prompt, max_output_tokens=1500):

        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt,
            config={
                "temperature": 0.2,
                "max_output_tokens": max_output_tokens
            }
        )

        if not response.text:
            raise ValueError("Gemini returned an empty response.")

        return response.text


# =====================================================
# 3. TAVILY RESEARCH SEARCH
# =====================================================

class ResearchSearch:

    def __init__(self, client):
        self.client = client

    def search(self, query, max_results=5):

        response = self.client.search(
            query=query,
            max_results=max_results,
            search_depth="basic"
        )

        sources = []

        for i, item in enumerate(
            response.get("results", []), 1
        ):
            sources.append({
                "id": f"S{i}",
                "title": item.get("title", "Untitled"),
                "url": item.get("url", ""),
                "content": item.get("content", "")
            })

        return sources


# =====================================================
# 4. SEVEN AI AGENTS
# =====================================================

class OrchestratorAgent:

    def __init__(self, gemini):
        self.gemini = gemini

    def plan(self, topic):

        prompt = f"""
        You are the Orchestrator Agent.

        Create a structured research plan for:
        {topic}

        Include:
        1. Research objectives
        2. Important questions
        3. Analysis areas
        4. Expected report sections

        Return a clear structured plan.
        """

        return self.gemini.generate(prompt)


class ResearchAgent:

    def __init__(self, search):
        self.search_engine = search

    def research(self, topic):

        return self.search_engine.search(
            topic,
            max_results=7
        )


class AnalysisAgent:

    def __init__(self, gemini):
        self.gemini = gemini

    def analyze(self, topic, sources):

        evidence = json.dumps(sources, ensure_ascii=False)

        prompt = f"""
        You are an AI Research Analysis Agent.

        Topic: {topic}

        Research evidence:
        {evidence}

        Analyze:
        1. Major findings
        2. Important patterns
        3. Advantages
        4. Limitations
        5. Evidence gaps

        Use source IDs when referencing evidence.
        Do not invent facts.
        """

        return self.gemini.generate(prompt, 2000)


class FactCheckerAgent:

    def __init__(self, gemini):
        self.gemini = gemini

    def check(self, topic, analysis, sources):

        prompt = f"""
        You are a Fact Checking Agent.

        Topic: {topic}

        Analysis:
        {analysis}

        Available sources:
        {json.dumps(sources, ensure_ascii=False)}

        Review important claims.

        Classify claims as:
        - Supported
        - Partially supported
        - Unsupported
        - Contradicted

        Explain limitations.
        Never claim that an AI check guarantees truth.
        """

        return self.gemini.generate(prompt, 2000)


class SummarizerAgent:

    def __init__(self, gemini):
        self.gemini = gemini

    def summarize(self, analysis, fact_check):

        prompt = f"""
        You are a Research Summarization Agent.

        Analysis:
        {analysis}

        Fact checking:
        {fact_check}

        Create a concise executive summary.
        Preserve important caveats and source references.
        """

        return self.gemini.generate(prompt, 1200)


class WriterAgent:

    def __init__(self, gemini):
        self.gemini = gemini

    def write_report(
        self, topic, plan, analysis,
        fact_check, summary, sources
    ):

        prompt = f"""
        You are a Professional Research Report Writer.

        Research topic: {topic}

        Research plan:
        {plan}

        Analysis:
        {analysis}

        Fact check:
        {fact_check}

        Executive summary:
        {summary}

        Sources:
        {json.dumps(sources, ensure_ascii=False)}

        Prepare a professional report with:

        1. Title
        2. Executive Summary
        3. Research Objectives
        4. Methodology
        5. Major Findings
        6. Detailed Analysis
        7. Fact Checking and Limitations
        8. Conclusion
        9. References

        Cite available source IDs.
        Never invent references.
        """

        return self.gemini.generate(prompt, 4000)


class QualityAgent:

    def __init__(self, gemini):
        self.gemini = gemini

    def review(self, report):

        prompt = f"""
        You are the Quality Review Agent.

        Review this research report:

        {report}

        Check:
        1. Completeness
        2. Clarity
        3. Logical consistency
        4. Source references
        5. Unsupported claims
        6. Limitations

        Give a quality assessment and improvements.
        Finish with exactly one status:
        APPROVED or NEEDS MAJOR REVISION.
        """

        return self.gemini.generate(prompt, 1500)


# =====================================================
# 5. DOCUMENT LOADER
# =====================================================

class DocumentLoader:

    def load(self, file_path):

        extension = os.path.splitext(file_path)[1].lower()

        if extension == ".txt":

            with open(file_path, "r", encoding="utf-8") as file:
                return file.read()

        elif extension == ".pdf":

            reader = PdfReader(file_path)

            return "\n".join(
                page.extract_text() or ""
                for page in reader.pages
            )

        elif extension == ".docx":

            document = Document(file_path)

            return "\n".join(
                paragraph.text
                for paragraph in document.paragraphs
            )

        else:
            raise ValueError("Only PDF, TXT and DOCX are supported.")


# =====================================================
# 6. TEXT CHUNKER
# =====================================================

class TextChunker:

    def __init__(self, chunk_size=500, overlap=100):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def split(self, text):

        words = text.split()
        chunks = []

        step = self.chunk_size - self.overlap

        if step <= 0:
            raise ValueError("Overlap must be smaller than chunk size.")

        for start in range(0, len(words), step):

            chunk = words[start:start + self.chunk_size]

            if chunk:
                chunks.append(" ".join(chunk))

        return chunks


# =====================================================
# 7. FAISS VECTOR STORE
# =====================================================

class FAISSVectorStore:

    def __init__(self, embedding_model):

        self.embedding_model = embedding_model
        self.dimension = embedding_model.get_sentence_embedding_dimension()

        self.index = faiss.IndexFlatIP(self.dimension)
        self.metadata = []

    def add_documents(self, chunks, source_name):

        if not chunks:
            return

        embeddings = self.embedding_model.encode(
            chunks,
            convert_to_numpy=True
        )

        embeddings = np.asarray(
            embeddings,
            dtype="float32"
        )

        faiss.normalize_L2(embeddings)

        self.index.add(embeddings)

        for i, chunk in enumerate(chunks):
            self.metadata.append({
                "text": chunk,
                "source": source_name,
                "chunk": i + 1
            })

    def search(self, query, top_k=4):

        if self.index.ntotal == 0:
            return []

        embedding = self.embedding_model.encode(
            [query],
            convert_to_numpy=True
        )

        embedding = np.asarray(
            embedding,
            dtype="float32"
        )

        faiss.normalize_L2(embedding)

        scores, indices = self.index.search(
            embedding,
            min(top_k, self.index.ntotal)
        )

        results = []

        for score, index in zip(scores[0], indices[0]):

            if index < 0:
                continue

            item = self.metadata[index].copy()
            item["score"] = float(score)

            results.append(item)

        return results


# =====================================================
# 8. RAG ASSISTANT
# =====================================================

class ResearchRAG:

    def __init__(
        self,
        loader,
        chunker,
        vector_store,
        gemini
    ):

        self.loader = loader
        self.chunker = chunker
        self.vector_store = vector_store
        self.gemini = gemini

    def add_document(self, file_path):

        text = self.loader.load(file_path)

        if not text.strip():
            raise ValueError("Document contains no extractable text.")

        chunks = self.chunker.split(text)

        self.vector_store.add_documents(
            chunks,
            os.path.basename(file_path)
        )

        return len(chunks)

    def ask(self, question):

        results = self.vector_store.search(
            question,
            top_k=4
        )

        if not results:
            return "No relevant documents found. Upload a document first."

        context = "\n\n".join(
            f"Source: {item['source']}\n{item['text']}"
            for item in results
        )

        prompt = f"""
        Answer the question using only the supplied context.

        Context:
        {context}

        Question:
        {question}

        If the answer is not available in the context,
        clearly say so.

        Mention the document sources used.
        """

        return self.gemini.generate(prompt, 1500)


# =====================================================
# 9. LANGGRAPH STATE
# =====================================================

class ResearchState(TypedDict, total=False):

    topic: str
    plan: str
    sources: List[Dict[str, Any]]
    analysis: str
    fact_check: str
    summary: str
    final_report: str
    quality_review: str
    revision_count: int
    status: str


# =====================================================
# 10. INITIALIZE API CLIENTS AND AGENTS
# =====================================================

gemini_client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

tavily_client = TavilyClient(
    api_key=os.environ["TAVILY_API_KEY"]
)

gemini_service = GeminiService(
    gemini_client,
    GEMINI_MODEL
)

research_search = ResearchSearch(tavily_client)

orchestrator = OrchestratorAgent(gemini_service)
research_agent = ResearchAgent(research_search)
analysis_agent = AnalysisAgent(gemini_service)
fact_checker = FactCheckerAgent(gemini_service)
summarizer = SummarizerAgent(gemini_service)
writer_agent = WriterAgent(gemini_service)
quality_agent = QualityAgent(gemini_service)


# =====================================================
# 11. LANGGRAPH WORKFLOW NODES
# =====================================================

def orchestrator_node(state):

    topic = state["topic"]

    return {
        "plan": orchestrator.plan(topic),
        "status": "Planning completed"
    }


def research_node(state):

    sources = research_agent.research(state["topic"])

    return {
        "sources": sources,
        "status": "Research completed"
    }


def analysis_node(state):

    analysis = analysis_agent.analyze(
        state["topic"],
        state.get("sources", [])
    )

    return {
        "analysis": analysis,
        "status": "Analysis completed"
    }


def fact_checker_node(state):

    result = fact_checker.check(
        state["topic"],
        state["analysis"],
        state.get("sources", [])
    )

    return {
        "fact_check": result,
        "status": "Fact checking completed"
    }


def summarizer_node(state):

    summary = summarizer.summarize(
        state["analysis"],
        state["fact_check"]
    )

    return {
        "summary": summary,
        "status": "Summary completed"
    }


def writer_node(state):

    report = writer_agent.write_report(
        state["topic"],
        state["plan"],
        state["analysis"],
        state["fact_check"],
        state["summary"],
        state.get("sources", [])
    )

    return {
        "final_report": report,
        "status": "Report generated"
    }


def quality_node(state):

    review = quality_agent.review(
        state["final_report"]
    )

    return {
        "quality_review": review,
        "status": "Quality review completed"
    }


def quality_decision(state):

    review = state.get("quality_review", "").upper()

    if (
        "NEEDS MAJOR REVISION" in review
        and state.get("revision_count", 0) < 1
    ):
        return "revise"

    return "approved"


def revision_node(state):

    revised_report = gemini_service.generate(
        f"""
        Improve this research report using the review.

        Report:
        {state['final_report']}

        Review:
        {state['quality_review']}

        Preserve evidence and references.
        Do not invent facts.
        """,
        4000
    )

    return {
        "final_report": revised_report,
        "revision_count": state.get("revision_count", 0) + 1,
        "status": "Revision completed"
    }


# =====================================================
# 12. BUILD LANGGRAPH
# =====================================================

workflow_builder = StateGraph(ResearchState)

workflow_builder.add_node("orchestrator", orchestrator_node)
workflow_builder.add_node("research", research_node)
workflow_builder.add_node("analysis", analysis_node)
workflow_builder.add_node("fact_checker", fact_checker_node)
workflow_builder.add_node("summarizer", summarizer_node)
workflow_builder.add_node("writer", writer_node)
workflow_builder.add_node("quality", quality_node)
workflow_builder.add_node("revision", revision_node)

workflow_builder.add_edge(START, "orchestrator")
workflow_builder.add_edge("orchestrator", "research")
workflow_builder.add_edge("research", "analysis")
workflow_builder.add_edge("analysis", "fact_checker")
workflow_builder.add_edge("fact_checker", "summarizer")
workflow_builder.add_edge("summarizer", "writer")
workflow_builder.add_edge("writer", "quality")

workflow_builder.add_conditional_edges(
    "quality",
    quality_decision,
    {
        "approved": END,
        "revise": "revision"
    }
)

workflow_builder.add_edge("revision", END)

research_workflow = workflow_builder.compile()


# =====================================================
# 13. INITIALIZE RAG
# =====================================================

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

document_loader = DocumentLoader()

text_chunker = TextChunker(
    chunk_size=500,
    overlap=100
)

vector_store = FAISSVectorStore(
    embedding_model
)

rag_assistant = ResearchRAG(
    document_loader,
    text_chunker,
    vector_store,
    gemini_service
)


print("ResearchMind AI backend initialized successfully.")

Writing /content/ResearchMind_AI/researchmind_core.py


In [ ]:
import os
import py_compile

backend_path = "/content/ResearchMind_AI/researchmind_core.py"

py_compile.compile(
    backend_path,
    doraise=True
)

print("Backend file:", os.path.exists(backend_path))
print("Backend size:", os.path.getsize(backend_path), "bytes")
print("Python syntax: PASSED")

Backend file: True
Backend size: 16369 bytes
Python syntax: PASSED


In [ ]:
import sys
import importlib

PROJECT_DIR = "/content/ResearchMind_AI"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import researchmind_core as core

checks = {
    "Gemini Service": hasattr(core, "gemini_service"),
    "Tavily Search": hasattr(core, "research_search"),
    "Seven Agents": all(
        hasattr(core, name)
        for name in [
            "orchestrator",
            "research_agent",
            "analysis_agent",
            "fact_checker",
            "summarizer",
            "writer_agent",
            "quality_agent"
        ]
    ),
    "RAG Assistant": hasattr(core, "rag_assistant"),
    "LangGraph Workflow": hasattr(core, "research_workflow")
}

print("========== BACKEND TEST ==========")

for name, result in checks.items():
    print(name, ":", "PASSED" if result else "FAILED")

assert all(checks.values()), "Backend validation failed!"

print("\nBACKEND VALIDATION: PASSED")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ResearchMind AI backend initialized successfully.
========== BACKEND TEST ==========
Gemini Service : PASSED
Tavily Search : PASSED
Seven Agents : PASSED
RAG Assistant : PASSED
LangGraph Workflow : PASSED

BACKEND VALIDATION: PASSED


/content/ResearchMind_AI/researchmind_core.py:372: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = embedding_model.get_sentence_embedding_dimension()


In [ ]:
import sys
sys.path.insert(0, "/content/ResearchMind_AI")

import researchmind_core

print("RESEARCHMIND AI BACKEND: SUCCESS")

RESEARCHMIND AI BACKEND: SUCCESS


In [ ]:
import os

PROJECT_DIR = "/content/ResearchMind_AI"

print("Backend:", os.path.exists(
    f"{PROJECT_DIR}/researchmind_core.py"
))

print("App file:", os.path.exists(
    f"{PROJECT_DIR}/app.py"
))

print("Exports folder:", os.path.isdir(
    f"{PROJECT_DIR}/exports"
))

print("Uploads folder:", os.path.isdir(
    f"{PROJECT_DIR}/uploads"
))

Backend: True
App file: False
Exports folder: True
Uploads folder: True


In [ ]:
%%writefile /content/ResearchMind_AI/app.py

import os
import sys
import hashlib
from datetime import datetime
from pathlib import Path
from xml.sax.saxutils import escape

import streamlit as st

PROJECT_DIR = "/content/ResearchMind_AI"
UPLOAD_DIR = os.path.join(PROJECT_DIR, "uploads")
EXPORT_DIR = os.path.join(PROJECT_DIR, "exports")

os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

sys.path.insert(0, PROJECT_DIR)

# Load Streamlit Cloud secrets if available
try:
    if "GEMINI_API_KEY" not in os.environ:
        os.environ["GEMINI_API_KEY"] = st.secrets["GEMINI_API_KEY"]

    if "TAVILY_API_KEY" not in os.environ:
        os.environ["TAVILY_API_KEY"] = st.secrets["TAVILY_API_KEY"]
except Exception:
    pass

st.set_page_config(
    page_title="ResearchMind AI",
    page_icon="🧠",
    layout="wide"
)

# -------------------- CUSTOM DESIGN --------------------

st.markdown("""
<style>
.stApp {
    background: #0b1120;
    color: #e5e7eb;
}
[data-testid="stSidebar"] {
    background: #111827;
}
.main-title {
    font-size: 38px;
    font-weight: 800;
    color: #8bafff;
}
.subtitle {
    color: #9ca3af;
    font-size: 15px;
}
.metric-card {
    background: #172238;
    padding: 18px;
    border-radius: 12px;
    border: 1px solid #263650;
}
div.stButton > button {
    border-radius: 9px;
    font-weight: 600;
}
</style>
""", unsafe_allow_html=True)

# -------------------- BACKEND --------------------

@st.cache_resource
def load_backend():
    import researchmind_core
    return researchmind_core

try:
    core = load_backend()
except Exception as error:
    st.error("Backend initialization failed.")
    st.exception(error)
    st.stop()

# -------------------- SESSION STATE --------------------

defaults = {
    "history": [],
    "latest_result": None,
    "latest_topic": "",
    "uploaded_hashes": set(),
    "rag_assistant": None,
    "rag_messages": []
}

for key, value in defaults.items():
    if key not in st.session_state:
        st.session_state[key] = value

# Create a separate vector store for this user's session
if st.session_state.rag_assistant is None:
    session_vector_store = core.FAISSVectorStore(
        core.embedding_model
    )

    st.session_state.rag_assistant = core.ResearchRAG(
        core.document_loader,
        core.text_chunker,
        session_vector_store,
        core.gemini_service
    )

rag = st.session_state.rag_assistant

# -------------------- HELPER FUNCTIONS --------------------

def make_markdown(topic, result):
    return f"""
# ResearchMind AI Research Report

**Topic:** {topic}

**Generated:** {datetime.now().strftime("%d-%m-%Y %H:%M")}

---

## Final Report

{result.get("final_report", "")}

---

## Analysis

{result.get("analysis", "")}

---

## Fact Check

{result.get("fact_check", "")}

---

## Summary

{result.get("summary", "")}

---

## Quality Review

{result.get("quality_review", "")}

---

## Sources

{format_sources(result.get("sources", []))}
"""


def format_sources(sources):
    if not sources:
        return "No sources available."

    lines = []
    for index, source in enumerate(sources, 1):
        title = source.get("title", "Untitled")
        url = source.get("url", "")
        lines.append(f"{index}. [{title}]({url})")

    return "\n".join(lines)


def make_pdf(markdown_text):
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer
    )
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.units import mm
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_LEFT
    from io import BytesIO

    buffer = BytesIO()
    doc = SimpleDocTemplate(
        buffer,
        pagesize=A4,
        rightMargin=18 * mm,
        leftMargin=18 * mm,
        topMargin=18 * mm,
        bottomMargin=18 * mm
    )

    styles = getSampleStyleSheet()
    styles["Title"].textColor = colors.HexColor("#244b87")
    styles["Heading2"].textColor = colors.HexColor("#244b87")

    story = []

    for line in markdown_text.splitlines():
        line = line.strip()

        if not line or line == "---":
            story.append(Spacer(1, 7))
            continue

        if line.startswith("# "):
            story.append(Paragraph(
                escape(line[2:]), styles["Title"]
            ))
        elif line.startswith("## "):
            story.append(Paragraph(
                escape(line[3:]), styles["Heading2"]
            ))
        else:
            clean = line.replace("**", "")
            clean = clean.replace("`", "")
            clean = clean.replace("[", "").replace("]", "")
            clean = clean.replace("(", " ").replace(")", "")
            story.append(Paragraph(
                escape(clean), styles["BodyText"]
            ))
            story.append(Spacer(1, 5))

    doc.build(story)
    return buffer.getvalue()


def show_sources(sources):
    if not sources:
        st.info("No research sources returned.")
        return

    for i, source in enumerate(sources, 1):
        with st.expander(
            f"{i}. {source.get('title', 'Untitled source')}"
        ):
            st.write(source.get("content", ""))
            if source.get("url"):
                st.markdown(
                    f"[Open source]({source['url']})"
                )

# -------------------- SIDEBAR --------------------

with st.sidebar:
    st.markdown("## 🧠 ResearchMind AI")
    st.caption("Multi-Agent AI Research Assistant")

    st.divider()

    st.markdown("### System Status")

    st.success("Backend initialized")

    if os.environ.get("GEMINI_API_KEY"):
        st.success("Gemini API configured")
    else:
        st.error("Gemini API key missing")

    if os.environ.get("TAVILY_API_KEY"):
        st.success("Tavily API configured")
    else:
        st.error("Tavily API key missing")

    st.info("RAG assistant initialized")

    st.divider()

    st.markdown("### Seven AI Agents")
    agent_names = [
        "1. Orchestrator",
        "2. Research",
        "3. Analysis",
        "4. Fact Checker",
        "5. Summarizer",
        "6. Writer",
        "7. Quality"
    ]

    for agent in agent_names:
        st.write("🤖 " + agent)

    st.divider()
    st.caption("ResearchMind AI | Prototype")

# -------------------- HEADER --------------------

st.markdown(
    '<div class="main-title">🧠 ResearchMind AI</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">Your intelligent multi-agent research workspace</div>',
    unsafe_allow_html=True
)

st.write("")

# -------------------- DASHBOARD --------------------

tab_research, tab_documents, tab_history, tab_about = st.tabs([
    "🔍 Research",
    "📚 Document Chat",
    "🕘 History",
    "ℹ️ About"
])

# =====================================================
# TAB 1 - RESEARCH
# =====================================================

with tab_research:

    st.subheader("AI Research Workspace")

    topic = st.text_area(
        "Enter your research topic",
        placeholder=(
            "Example: Explain RAG architecture, "
            "applications, advantages and limitations."
        ),
        height=120
    )

    run_research = st.button(
        "🚀 Generate Research Report",
        type="primary",
        use_container_width=True
    )

    if run_research:

        if not topic.strip():
            st.warning("Please enter a research topic.")

        elif not os.environ.get("GEMINI_API_KEY"):
            st.error("Gemini API key is not configured.")

        elif not os.environ.get("TAVILY_API_KEY"):
            st.error("Tavily API key is not configured.")

        else:
            try:
                with st.spinner(
                    "ResearchMind AI agents are working..."
                ):
                    result = core.research_workflow.invoke({
                        "topic": topic.strip(),
                        "revision_count": 0
                    })

                st.session_state.latest_result = result
                st.session_state.latest_topic = topic.strip()

                st.session_state.history.insert(0, {
                    "topic": topic.strip(),
                    "result": result,
                    "time": datetime.now().strftime(
                        "%d-%m-%Y %H:%M"
                    )
                })

                st.success("Research workflow completed!")

            except Exception as error:
                st.error("Research workflow failed.")
                st.exception(error)

    # Display latest research
    result = st.session_state.latest_result

    if result:

        st.divider()
        st.subheader("Research Results")

        stages = [
            "Orchestrator",
            "Research",
            "Analysis",
            "Fact Checker",
            "Summarizer",
            "Writer",
            "Quality"
        ]

        st.markdown("### Agent Workflow")

        cols = st.columns(4)

        for index, name in enumerate(stages):
            with cols[index % 4]:
                st.success("✓ " + name)

        st.caption(
            "Stages shown after the workflow returns successfully; "
            "these are not live streamed progress updates."
        )

        st.divider()

        result_tabs = st.tabs([
            "📄 Final Report",
            "📊 Analysis",
            "🔎 Fact Check",
            "📝 Summary",
            "🌐 Sources",
            "⭐ Quality"
        ])

        with result_tabs[0]:
            st.markdown(
                result.get("final_report", "No report generated.")
            )

        with result_tabs[1]:
            st.markdown(
                result.get("analysis", "No analysis available.")
            )

        with result_tabs[2]:
            st.markdown(
                result.get("fact_check", "No fact check available.")
            )

        with result_tabs[3]:
            st.markdown(
                result.get("summary", "No summary available.")
            )

        with result_tabs[4]:
            show_sources(result.get("sources", []))

        with result_tabs[5]:
            st.markdown(
                result.get("quality_review", "No quality review.")
            )

        st.divider()
        st.subheader("Export Research")

        markdown_report = make_markdown(
            st.session_state.latest_topic,
            result
        )

        col1, col2 = st.columns(2)

        with col1:
            st.download_button(
                "⬇️ Download Markdown",
                data=markdown_report,
                file_name="ResearchMind_Report.md",
                mime="text/markdown",
                use_container_width=True
            )

        with col2:
            try:
                pdf_data = make_pdf(markdown_report)

                st.download_button(
                    "⬇️ Download PDF",
                    data=pdf_data,
                    file_name="ResearchMind_Report.pdf",
                    mime="application/pdf",
                    use_container_width=True
                )
            except Exception as error:
                st.warning(f"PDF export unavailable: {error}")

# =====================================================
# TAB 2 - DOCUMENT CHAT / RAG
# =====================================================

with tab_documents:

    st.subheader("📚 Chat with Your Documents")

    st.write(
        "Upload PDF, TXT or DOCX files and ask questions "
        "based on their content."
    )

    uploaded_files = st.file_uploader(
        "Upload your documents",
        type=["pdf", "txt", "docx"],
        accept_multiple_files=True
    )

    if uploaded_files:

        for uploaded_file in uploaded_files:

            file_bytes = uploaded_file.getvalue()
            file_hash = hashlib.sha256(file_bytes).hexdigest()

            if file_hash not in st.session_state.uploaded_hashes:

                extension = Path(uploaded_file.name).suffix.lower()
                safe_name = file_hash + extension
                file_path = os.path.join(UPLOAD_DIR, safe_name)

                try:
                    with open(file_path, "wb") as file:
                        file.write(file_bytes)

                    with st.spinner(
                        f"Processing {uploaded_file.name}..."
                    ):
                        chunk_count = rag.add_document(file_path)

                    st.session_state.uploaded_hashes.add(file_hash)

                    st.success(
                        f"{uploaded_file.name}: "
                        f"{chunk_count} chunks indexed."
                    )

                except Exception as error:
                    st.error(
                        f"Could not process {uploaded_file.name}"
                    )
                    st.exception(error)

            else:
                st.info(
                    f"{uploaded_file.name} is already indexed "
                    "in this session."
                )

    st.divider()
    st.markdown("### Ask a Question")

    with st.form("document_question_form"):
        question = st.text_input(
            "Your question",
            placeholder="What does this document explain?"
        )

        ask_button = st.form_submit_button(
            "💬 Ask ResearchMind AI"
        )

    if ask_button:

        if not question.strip():
            st.warning("Enter a question first.")

        elif not st.session_state.uploaded_hashes:
            st.warning("Upload and process a document first.")

        else:
            try:
                with st.spinner("Searching your documents..."):
                    answer = rag.ask(question.strip())

                st.session_state.rag_messages.append({
                    "question": question.strip(),
                    "answer": answer
                })

            except Exception as error:
                st.error("Document question failed.")
                st.exception(error)

    for message in reversed(st.session_state.rag_messages):
        st.markdown("**You:** " + message["question"])
        st.markdown("**ResearchMind AI:**")
        st.write(message["answer"])
        st.divider()

    st.caption(
        "Uploaded documents are held in the current app process/session. "
        "They are not permanently saved in a persistent vector database."
    )

# =====================================================
# TAB 3 - HISTORY
# =====================================================

with tab_history:

    st.subheader("🕘 Research History")

    if not st.session_state.history:
        st.info("Your research history will appear here.")

    else:
        for index, item in enumerate(
            st.session_state.history
        ):
            with st.expander(
                f"{item['topic']} | {item['time']}"
            ):
                st.markdown(
                    item["result"].get(
                        "final_report",
                        "No report available."
                    )
                )

                history_md = make_markdown(
                    item["topic"],
                    item["result"]
                )

                st.download_button(
                    "Download this report",
                    data=history_md,
                    file_name=f"Research_Report_{index + 1}.md",
                    mime="text/markdown",
                    key=f"history_download_{index}"
                )

# =====================================================
# TAB 4 - ABOUT
# =====================================================

with tab_about:

    st.subheader("About ResearchMind AI")

    st.write("""
    ResearchMind AI is a multi-agent research assistant designed
    to automate research workflows.

    **Core technologies**

    - Python
    - Google Gemini API
    - Tavily Search API
    - LangGraph
    - Sentence Transformers
    - FAISS Vector Database
    - Streamlit

    **Seven-agent architecture**

    1. Orchestrator Agent
    2. Research Agent
    3. Analysis Agent
    4. Fact Checker Agent
    5. Summarizer Agent
    6. Writer Agent
    7. Quality Agent

    **Main capabilities**

    - AI-powered web research
    - Multi-agent report generation
    - Fact checking
    - Document-based RAG question answering
    - PDF and Markdown report exports
    """)

    st.info(
        "ResearchMind AI is a prototype. "
        "Always independently verify research findings and sources."
    )

st.divider()

st.caption(
    "ResearchMind AI | Built with Python, Gemini, LangGraph, "
    "FAISS and Streamlit"
)

Overwriting /content/ResearchMind_AI/app.py


In [ ]:
import os
import py_compile

APP_FILE = "/content/ResearchMind_AI/app.py"

print("App file exists:", os.path.exists(APP_FILE))

py_compile.compile(APP_FILE, doraise=True)

print("APP SYNTAX CHECK: PASSED")

App file exists: True
APP SYNTAX CHECK: PASSED


In [ ]:
import os
import sys
import time
import subprocess

PROJECT_DIR = "/content/ResearchMind_AI"
LOG_FILE = "/content/ResearchMind_AI/streamlit.log"

# Stop the previous app process if it exists
try:
    if app_process.poll() is None:
        app_process.terminate()
        app_process.wait(timeout=5)
except (NameError, AttributeError):
    pass
except Exception:
    pass

# Start Streamlit
log_handle = open(LOG_FILE, "w")

app_process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run",
        f"{PROJECT_DIR}/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.fileWatcherType", "none"
    ],
    cwd=PROJECT_DIR,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=os.environ.copy()
)

time.sleep(12)

if app_process.poll() is None:
    print("RESEARCHMIND AI SERVER: RUNNING")
    print("Process ID:", app_process.pid)

    from google.colab import output
    output.serve_kernel_port_as_iframe(8501)
else:
    print("SERVER FAILED TO START")
    with open(LOG_FILE, "r") as f:
        print(f.read()[-5000:])

RESEARCHMIND AI SERVER: RUNNING
Process ID: 10865


<IPython.core.display.Javascript object>

In [ ]:
%%writefile /content/ResearchMind_AI/uploads/sample_ai_document.txt

Artificial Intelligence is a technology that enables machines
to perform tasks that normally require human intelligence.

Machine Learning is a subset of Artificial Intelligence.
Deep Learning uses neural networks with multiple layers.

Generative AI can create text, images, audio and code.
Large Language Models are trained on large amounts of text data.

Retrieval Augmented Generation combines document retrieval
with language model generation.

RAG improves answers by retrieving relevant information
from external knowledge sources.

Writing /content/ResearchMind_AI/uploads/sample_ai_document.txt


In [ ]:
What is Retrieval Augmented Generation and how does it work?

Object `work` not found.


In [ ]:
%%writefile /content/ResearchMind_AI/README.md

# 🧠 ResearchMind AI – Multi-Agent AI Research Assistant

An AI-powered multi-agent research assistant built using Python, Google Gemini, Tavily Search, LangGraph, FAISS, RAG and Streamlit.

## Features

- Seven specialized AI agents
- Web research using Tavily
- AI analysis and fact checking
- Research summaries and report generation
- PDF, TXT and DOCX document upload
- RAG-based document question answering
- FAISS vector search
- PDF and Markdown report export
- Streamlit dashboard

## Tech Stack

Python, Google Gemini, Tavily, LangGraph, Sentence Transformers, FAISS, Streamlit, RAG.

## Installation

```bash
git clone https://github.com/YOUR-USERNAME/ResearchMind-AI.git
cd ResearchMind-AI
pip install -r requirements.txt
streamlit run app.py

Writing /content/ResearchMind_AI/README.md


In [ ]:
%%writefile /content/ResearchMind_AI/requirements.txt

streamlit
google-genai
tavily-python
sentence-transformers
faiss-cpu
numpy
langgraph
pypdf
python-docx
reportlab

Writing /content/ResearchMind_AI/requirements.txt
